In [ ]:
!nvidia-smi

Sat Apr 18 11:32:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive, userdata, files
import os
import sys
import shutil
import time
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
from torchvision.models import GoogLeNet_Weights
from torchvision.models import ResNet18_Weights
import torchvision.transforms as transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

drive.mount('/content/drive')
output_path = Path("/content/drive/MyDrive/The_Masters/Text_Analytics/trained_models")
sentence_emb_stem = Path("/content/drive/MyDrive/The_Masters/Text_Analytics/embeddings")
images_path = Path("/content/drive/MyDrive/The_Masters/Text_Analytics/embeddings/images_master.pt")
images = torch.load(images_path, map_location='cpu')


Mounted at /content/drive


In [3]:
class Dataset_A(Dataset):
    def __init__(self, images : torch.Tensor, sentences : torch.Tensor, transformation):

        self.images = images
        self.sentences = sentences
        self.transformation = transformation
        if len(self.images) != len(self.sentences):
            raise ValueError(f'Mismatch in embedding lengths. IMG_EMB: {len(self.images)} | SENTENCE_EMB: {len(self.sentences)}')
        print('Dataloader initialisation complete')
        print(f'Sentence embeddings:\n>>> Shape: {self.sentences.shape}\n>>> Type: {self.sentences.dtype}')
        print(f'Image embeddings:\n>>> Shape: {self.images.shape}\n>>> Type: {self.images.dtype}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_emb = self.images[idx]
        if img_emb is None:
            raise ValueError(f'Img embedding for {idx} is none')
        img_emb_t = self.transformation(img_emb)
        sentence_emb = self.sentences[idx]
        return img_emb_t, sentence_emb

In [4]:
class EarlyStopping:
    # Adapted from https://medium.com/biased-algorithms/a-practical-guide-to-implementing-early-stopping-in-pytorch-for-model-training-99a7cbd46e9d
    def __init__(self, patience : int, min_improvement : float = 0.0):
        self.patience = patience
        self.min_improvement = min_improvement
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop = False

    def check(self, val_loss : float):
        if self.best_loss is None or val_loss < self.best_loss - self.min_improvement:
            self.best_loss = val_loss
            self.no_improvement_count = 0 # Reset counter
        else:
            self.no_improvement_count +=1
            if self.no_improvement_count > self.patience:
                self.stop = True

In [5]:
class CosineLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pred_emb, truth_emb):
        return (1.0 - F.cosine_similarity(pred_emb, truth_emb, dim=1)).mean()

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def train_one_epoch(model, train_set, optimizer, criterion):
    model.train()
    total_training_loss = 0.0
    for img_emb, sentence_emb in train_set:
        img_emb = img_emb.to(device)
        sentence_emb = sentence_emb.to(device)
        optimizer.zero_grad()
        outputs = model(img_emb)
        loss = criterion(outputs, sentence_emb)
        loss.backward()
        optimizer.step()
        total_training_loss += loss.item()
    return total_training_loss

def validation(model, val_loader, criterion):
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for img_emb, sentence_emb in val_loader:
            img_emb = img_emb.to(device)
            sentence_emb = sentence_emb.to(device)
            outputs = model(img_emb)
            loss = criterion(outputs, sentence_emb)
            total_val_loss += loss.item()
    return total_val_loss

def train(model, train_set, val_set, epochs, learning_rate, criterion):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.1, patience =2, min_lr = 1e-8
    )
    early_stopping = EarlyStopping(patience=4)
    train_losses = []
    val_losses = []
    start = time.time()
    for epoch in range(epochs):
        total_train_loss = train_one_epoch(model, train_set, optimizer, criterion)
        avg_train_loss = total_train_loss / len(train_set)
        train_losses.append(avg_train_loss)
        total_val_loss = validation(model, val_set, criterion)
        avg_val_loss = total_val_loss / len(val_set)
        scheduler.step(avg_val_loss)
        val_losses.append(avg_val_loss)
        early_stopping.check(avg_val_loss)
        print(f'Epoch: {epoch+1}/{epochs} | Train Loss: {round(avg_train_loss,6)} | Test Loss: {round(avg_val_loss,6)}')
        if early_stopping.stop == True:
            print('Early Stopping Activated!')
            break
    total_time = time.time() - start
    loss_type = criterion.__class__.__name__
    return train_losses, val_losses, model, total_time, loss_type

In [7]:
class CNNStride(nn.Module):
    def __init__(self, output_dims : int, dropout : int = 0.5):
        super().__init__()

        self.convolution = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size = 3, stride=2, padding=1), # Convolutional Layer 1
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, kernel_size = 3, stride=2, padding=1), # Convolutional Layer 2
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size = 3, stride=2, padding=1), # Convolutional Layer 3
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size = 3, stride=2, padding=1), # Convolutional Layer 4
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128),

            nn.AdaptiveAvgPool2d((3,3))
        )
        self.linear = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*3*3, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, output_dims)
        )

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        x = self.convolution(x)
        x = self.linear(x)
        return x

In [8]:
class CNNPool(nn.Module):
    def __init__(self, output_dims : int, dropout : int = 0.5):
        super().__init__()

        self.convolution = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size = 3, padding = 1), # Convolutional Layer 1
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),

            nn.Conv2d(16, 32, kernel_size = 3, padding = 1 ), # Convolutional Layer 2
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1), # Convolutional Layer 3
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1), # Convolutional Layer 4
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),

            nn.AdaptiveAvgPool2d((3, 3))
        )

        self.linear = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*3*3, 2500),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(2500, output_dims)
    )

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        x = self.convolution(x)
        x = self.linear(x)
        return x

In [9]:
class ResnetDSA(nn.Module):

    def __init__(self, output_dims: int, dropout: float = 0.5):
        super().__init__()

        self.convolution = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.convolution.fc = nn.Identity()
        self.custom_linear = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, output_dims)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.convolution(x)
        x = self.custom_linear(x)
        return x

In [10]:
class CNN2Layer(nn.Module):
    def __init__(self, output_dims: int, dropout: float = 0.5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                 #  64×64×32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                 #  32×32×64
        )

        self.avgpool = nn.AdaptiveAvgPool2d((4, 4))     #  4×4×64 = 1024  (32÷4 exact, MPS-safe)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, output_dims),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

In [11]:
class GoogleNet(nn.Module):

    def __init__(self, output_dims:int, dropout : int = 0.5):
        super().__init__()
        # Inputs : [batch_size, 3,224,224]
        self.convolution = models.googlenet(weights=GoogLeNet_Weights.DEFAULT)
        self.convolution.fc = nn.Identity()
        if self.convolution.aux_logits == True:
            raise ValueError('Aux_Logits still active')

        self.custom_linear = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, output_dims)
        )

    def forward(self, x : torch.Tensor) -> torch.Tensor:
        x = self.convolution(x)
        x = self.custom_linear(x)
        return x

In [12]:
transform_fs = transforms.Compose([
                transforms.Resize((128,128)),
                transforms.Normalize((0.5, 0.5,0.5), (0.5, 0.5, 0.5))
        ])

transform_pt = transforms.Compose([
                transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])

run_variations = [
        # Vary model size
    {'model' : CNN2Layer, 'loss': CosineLoss(), 'embedding': 'sbert_emb_master.pt', 'output_dims' : 384, 'transform' : transform_fs, 'emb' : 'sbert'},
    {'model' : CNNPool, 'loss': CosineLoss(), 'embedding': 'sbert_emb_master.pt', 'output_dims' : 384, 'transform' : transform_fs, 'emb' : 'sbert'},
    {'model' : CNNStride, 'loss': CosineLoss(), 'embedding': 'sbert_emb_master.pt', 'output_dims' : 384, 'transform' : transform_fs, 'emb' : 'sbert'},
    {'model' : GoogleNet, 'loss': CosineLoss(), 'embedding': 'sbert_emb_master.pt', 'output_dims' : 384, 'transform' : transform_pt, 'emb' : 'sbert'},
    {'model' : ResnetDSA, 'loss': CosineLoss(), 'embedding': 'sbert_emb_master.pt', 'output_dims' : 384, 'transform' : transform_pt, 'emb' : 'sbert'},

        # Vary Embeddings for cosine loss
    {'model' : ResnetDSA, 'loss': CosineLoss(), 'embedding': 'TB_pooler_emb_master.pt', 'output_dims' : 312, 'transform' : transform_pt, 'emb' : 'TB_pooler'},
    {'model' : ResnetDSA, 'loss': CosineLoss(), 'embedding': 'TB_mean_emb_master.pt', 'output_dims' : 312, 'transform' : transform_pt, 'emb' : 'TB_mean'},
    {'model' : ResnetDSA, 'loss': CosineLoss(), 'embedding': 'B_pooler_emb_master.pt', 'output_dims' : 768, 'transform' : transform_pt, 'emb' : 'B_pooler'},
    {'model' : ResnetDSA, 'loss': CosineLoss(), 'embedding': 'B_mean_emb_master.pt', 'output_dims' : 768, 'transform' : transform_pt, 'emb' : 'B_mean'},
    {'model' : ResnetDSA, 'loss': CosineLoss(), 'embedding': 'P_wvec_emb_master.pt', 'output_dims' : 300, 'transform' : transform_pt, 'emb' : 'P_wvec'},

        # Vary Embeddings with MSE loss
    {'model' : ResnetDSA, 'loss': nn.MSELoss(), 'embedding': 'sbert_emb_master.pt', 'output_dims' : 384, 'transform' : transform_pt, 'emb' : 'sbert'},
    {'model' : ResnetDSA, 'loss': nn.MSELoss(), 'embedding': 'TB_pooler_emb_master.pt', 'output_dims' : 312, 'transform' : transform_pt, 'emb' : 'TB_pooler'},
    {'model' : ResnetDSA, 'loss': nn.MSELoss(), 'embedding': 'TB_mean_emb_master.pt', 'output_dims' : 312, 'transform' : transform_pt, 'emb' : 'TB_mean'},
    {'model' : ResnetDSA, 'loss': nn.MSELoss(), 'embedding': 'B_pooler_emb_master.pt', 'output_dims' : 768, 'transform' : transform_pt, 'emb' : 'B_pooler'},
    {'model' : ResnetDSA, 'loss': nn.MSELoss(), 'embedding': 'B_mean_emb_master.pt', 'output_dims' : 768, 'transform' : transform_pt, 'emb' : 'B_mean'},
    {'model' : ResnetDSA, 'loss': nn.MSELoss(), 'embedding': 'P_wvec_emb_master.pt', 'output_dims' : 300, 'transform' : transform_pt, 'emb' : 'P_wvec'}
]

In [13]:
def train_val_test_split(images):
    torch.manual_seed(42)
    idxs = torch.randperm(len(images)).tolist()
    splits = {
      'train_idxs' : idxs[:8000],
      'val_idxs' : idxs[8000:9000],
      'test_idxs' : idxs[9000:]}
    return splits

def run_cases(sentence_emb_stem : Path, images : torch.Tensor, run_variations : dict, save_to_path : Path):
    start = time.time()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not sentence_emb_stem.exists():
        raise FileNotFoundError(f'Sentence embeddings folder not found: {sentence_emb_stem}')
    print('Images file found')
    print('Sentence embeddings folder found.')
    splits = train_val_test_split(images)
    splits_file_path = save_to_path / 'splits.pt'
    torch.save(splits, splits_file_path)
    print(f'Splits saved successfully: {splits_file_path}')
    for idx, variation in enumerate(run_variations):
        sentence_emb_path = Path(sentence_emb_stem) / variation['embedding']
        if not sentence_emb_path.exists():
            raise FileNotFoundError(f'Sentence embedding file not found: {sentence_emb_path}')
        sentences = torch.load(sentence_emb_path)
        dataset = Dataset_A(images, sentences, variation['transform'])
        training, validation = Subset(dataset, splits['train_idxs']), Subset(dataset, splits['val_idxs'])
        train_loader = DataLoader(training, batch_size = 32, shuffle=True)
        val_loader = DataLoader(validation, batch_size = 32, shuffle=False)
        model = variation['model']
        output_dims = variation['output_dims']
        model = model(output_dims = output_dims)
        model = model.to(device)
        print('-------------------------------------------------------')
        print(f'Beginning training for {model.__class__.__name__}')
        print(f'>> Embedding Type: {variation['emb']}')
        print(f'>> Loss: {variation['loss'].__class__.__name__}')
        train_losses, val_losses, best_model, total_time, loss_type = train(model, train_loader, val_loader, epochs = 15, learning_rate = 1e-5, criterion = variation['loss'])
        model_name = f"{idx}_{best_model.__class__.__name__}_{variation['emb']}_{variation['loss'].__class__.__name__}.pt"
        output_path = Path(save_to_path) / model_name
        best_model_data = {
                    'model_architecture': best_model.__class__.__name__,
                    'parameters': best_model.state_dict(),
                    'loss_type' : loss_type,
                    'embedding_type': variation['emb'],
                    'total_time': total_time,
                    'train_losses' : train_losses,
                    'val_losses' : val_losses
                }
        torch.save(best_model_data, output_path)
        print(f'Successfully trained {model_name} in {total_time:.2f}s')
        print(f'Saved model to: {output_path}')
    end = time.time() - start
    print(f'Model training has been finished in {end/60:.2f} minutes')

In [14]:
run_cases(sentence_emb_stem, images, run_variations, output_path)

Images file found
Sentence embeddings folder found.
Splits saved successfully: /content/drive/MyDrive/The_Masters/Text_Analytics/trained_models/splits.pt
Dataloader initialisation complete
Sentence embeddings:
>>> Shape: torch.Size([10000, 384])
>>> Type: torch.float32
Image embeddings:
>>> Shape: torch.Size([10000, 3, 224, 224])
>>> Type: torch.float32
-------------------------------------------------------
Beginning training for CNN2Layer
>> Embedding Type: sbert
>> Loss: CosineLoss
Epoch: 1/15 | Train Loss: 0.54876 | Test Loss: 0.258781
Epoch: 2/15 | Train Loss: 0.334476 | Test Loss: 0.225866
Epoch: 3/15 | Train Loss: 0.292582 | Test Loss: 0.218732
Epoch: 4/15 | Train Loss: 0.273296 | Test Loss: 0.215218
Epoch: 5/15 | Train Loss: 0.261272 | Test Loss: 0.212671
Epoch: 6/15 | Train Loss: 0.253095 | Test Loss: 0.210285
Epoch: 7/15 | Train Loss: 0.246463 | Test Loss: 0.20781
Epoch: 8/15 | Train Loss: 0.24051 | Test Loss: 0.205006
Epoch: 9/15 | Train Loss: 0.235371 | Test Loss: 0.201908


100%|██████████| 49.7M/49.7M [00:00<00:00, 117MB/s]


-------------------------------------------------------
Beginning training for GoogleNet
>> Embedding Type: sbert
>> Loss: CosineLoss
Epoch: 1/15 | Train Loss: 0.407922 | Test Loss: 0.190586
Epoch: 2/15 | Train Loss: 0.189178 | Test Loss: 0.146117
Epoch: 3/15 | Train Loss: 0.149991 | Test Loss: 0.113549
Epoch: 4/15 | Train Loss: 0.123035 | Test Loss: 0.092488
Epoch: 5/15 | Train Loss: 0.105981 | Test Loss: 0.079884
Epoch: 6/15 | Train Loss: 0.094227 | Test Loss: 0.070644
Epoch: 7/15 | Train Loss: 0.085315 | Test Loss: 0.062987
Epoch: 8/15 | Train Loss: 0.077655 | Test Loss: 0.0561
Epoch: 9/15 | Train Loss: 0.070441 | Test Loss: 0.050134
Epoch: 10/15 | Train Loss: 0.064608 | Test Loss: 0.045221
Epoch: 11/15 | Train Loss: 0.059283 | Test Loss: 0.041164
Epoch: 12/15 | Train Loss: 0.055386 | Test Loss: 0.03809
Epoch: 13/15 | Train Loss: 0.05202 | Test Loss: 0.035976
Epoch: 14/15 | Train Loss: 0.049155 | Test Loss: 0.034162
Epoch: 15/15 | Train Loss: 0.047167 | Test Loss: 0.032772
Successfu

100%|██████████| 44.7M/44.7M [00:00<00:00, 176MB/s]


-------------------------------------------------------
Beginning training for ResnetDSA
>> Embedding Type: sbert
>> Loss: CosineLoss
Epoch: 1/15 | Train Loss: 0.575658 | Test Loss: 0.220693
Epoch: 2/15 | Train Loss: 0.294795 | Test Loss: 0.160924
Epoch: 3/15 | Train Loss: 0.236251 | Test Loss: 0.131986
Epoch: 4/15 | Train Loss: 0.201727 | Test Loss: 0.11077
Epoch: 5/15 | Train Loss: 0.177949 | Test Loss: 0.095327
Epoch: 6/15 | Train Loss: 0.159796 | Test Loss: 0.083824
Epoch: 7/15 | Train Loss: 0.145233 | Test Loss: 0.075121
Epoch: 8/15 | Train Loss: 0.134083 | Test Loss: 0.068545
Epoch: 9/15 | Train Loss: 0.124036 | Test Loss: 0.062772
Epoch: 10/15 | Train Loss: 0.116288 | Test Loss: 0.058364
Epoch: 11/15 | Train Loss: 0.109352 | Test Loss: 0.054482
Epoch: 12/15 | Train Loss: 0.103402 | Test Loss: 0.051661
Epoch: 13/15 | Train Loss: 0.09802 | Test Loss: 0.048432
Epoch: 14/15 | Train Loss: 0.093971 | Test Loss: 0.046314
Epoch: 15/15 | Train Loss: 0.089556 | Test Loss: 0.044333
Success